<a href="https://colab.research.google.com/github/syedmahmoodiagents/NLP/blob/main/seq2seq_rnn_tf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, SimpleRNN, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [2]:
english = [
    "i love mathematics",
    "mathematics is useful",
    "i love calculus",
    "calculus is great"
]

dutch = [
    "ik hou van wiskunde",
    "wiskunde is nuttig",
    "ik hou van calculus",
    "calculus is geweldig"
]

In [3]:
# Add START and END tokens
decoder_input_text = ["<start> " + s for s in dutch]
decoder_output_text = [s + " <end>" for s in dutch]

In [4]:
encoder_tokenizer = Tokenizer(filters="")
encoder_tokenizer.fit_on_texts(english)

decoder_tokenizer = Tokenizer(filters="")
decoder_tokenizer.fit_on_texts(decoder_input_text + decoder_output_text)

In [5]:
encoder_vocab_size = len(encoder_tokenizer.word_index) + 1
decoder_vocab_size = len(decoder_tokenizer.word_index) + 1

In [6]:
encoder_sequences = encoder_tokenizer.texts_to_sequences(english)

In [7]:
decoder_input_sequences = decoder_tokenizer.texts_to_sequences(decoder_input_text)
decoder_output_sequences = decoder_tokenizer.texts_to_sequences(decoder_output_text)

In [8]:
max_encoder_len = max(len(s) for s in encoder_sequences)
max_decoder_len = max(len(s) for s in decoder_input_sequences)

In [9]:
encoder_sequences = pad_sequences(
    encoder_sequences,
    maxlen=max_encoder_len,
    padding="post"
)

decoder_input_sequences = pad_sequences(
    decoder_input_sequences,
    maxlen=max_decoder_len,
    padding="post"
)

decoder_output_sequences = pad_sequences(
    decoder_output_sequences,
    maxlen=max_decoder_len,
    padding="post"
)

In [10]:
decoder_output_sequences = np.expand_dims(decoder_output_sequences, -1)

In [11]:
embedding_dim = 64
hidden_units = 128

In [12]:



encoder_inputs = Input(shape=(None,))

encoder_embedding = Embedding(
    encoder_vocab_size,
    embedding_dim
)(encoder_inputs)

encoder_rnn = SimpleRNN(
    hidden_units,
    return_state=True
)

encoder_outputs, encoder_state = encoder_rnn(
    encoder_embedding
)

In [13]:


decoder_inputs = Input(shape=(None,))

decoder_embedding_layer = Embedding(
    decoder_vocab_size,
    embedding_dim
)

decoder_embedding = decoder_embedding_layer(
    decoder_inputs
)

decoder_rnn = SimpleRNN(
    hidden_units,
    return_sequences=True,
    return_state=True
)

decoder_outputs, decoder_state = decoder_rnn(
    decoder_embedding,
    initial_state=encoder_state
)

decoder_dense = Dense(
    decoder_vocab_size,
    activation="softmax"
)

decoder_outputs = decoder_dense(decoder_outputs)

In [14]:


model = Model(
    [encoder_inputs, decoder_inputs],
    decoder_outputs
)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, None, 64)  │        512 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, None, 64)  │        704 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ simple_rnn          │ [(None, 128),     │     24,704 │ embedding[0][0]   │
│ (SimpleRNN)         │ (None, 128)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ simple_rnn_1        │ [(None, None,     │     24,704 │ embedding_1[0][0… │
│ (SimpleRNN)         │ 128), (None,      │            │ simple_rnn[0][1]  │
│                     │ 128)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, None, 11)  │      1,419 │ simple_rnn_1[0][… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 52,043 (203.29 KB)

 Trainable params: 52,043 (203.29 KB)

 Non-trainable params: 0 (0.00 B)

In [15]:
model.fit(
    [encoder_sequences, decoder_input_sequences],
    decoder_output_sequences,
    epochs=300,
    verbose=0
)

In [16]:
encoder_model = Model(
    encoder_inputs,
    encoder_state
)

In [17]:

decoder_state_input = Input(shape=(hidden_units,))

decoder_single_input = Input(shape=(1,))

decoder_embed = decoder_embedding_layer(
    decoder_single_input
)

decoder_output, decoder_state = decoder_rnn(
    decoder_embed,
    initial_state=decoder_state_input
)

decoder_output = decoder_dense(
    decoder_output
)

decoder_model = Model(
    [decoder_single_input, decoder_state_input],
    [decoder_output, decoder_state]
)

In [18]:
# Reverse Lookup Dictionaries
reverse_decoder_index = {
    index: word
    for word, index in decoder_tokenizer.word_index.items()
}

In [21]:
decoder_tokenizer.word_index

{'<start>': 1,
 'ik': 2,
 'hou': 3,
 'van': 4,
 'wiskunde': 5,
 'is': 6,
 'calculus': 7,
 '<end>': 8,
 'nuttig': 9,
 'geweldig': 10}

In [22]:
start_token = decoder_tokenizer.word_index["<start>"]
end_token = decoder_tokenizer.word_index["<end>"]

In [23]:

def translate(sentence):

    sequence = encoder_tokenizer.texts_to_sequences([sentence])

    sequence = pad_sequences(
        sequence,
        maxlen=max_encoder_len,
        padding="post"
    )

    state = encoder_model.predict(sequence, verbose=0)

    target_seq = np.array([[start_token]])

    translated_sentence = []

    while True:

        prediction, state = decoder_model.predict(
            [target_seq, state],
            verbose=0
        )

        predicted_id = np.argmax(prediction[0, 0])

        if predicted_id == 0:
            break

        word = reverse_decoder_index[predicted_id]

        if word == "end":
            break

        translated_sentence.append(word)

        target_seq = np.array([[predicted_id]])

    return " ".join(translated_sentence)



In [24]:
tests = [
    "i love mathematics",
    "i love calculus",
    "mathematics is useful",
    "calculus is great"
]

for sentence in tests:
    print("------------------------------------")
    print("English :", sentence)
    print("Dutch   :", translate(sentence))

------------------------------------
English : i love mathematics
Dutch   : ik hou van wiskunde <end>
------------------------------------
English : i love calculus
Dutch   : ik hou van calculus <end>
------------------------------------
English : mathematics is useful
Dutch   : wiskunde is nuttig <end>
------------------------------------
English : calculus is great
Dutch   : calculus is geweldig <end>
